In [10]:
import mappy as mp
import sys
import statistics
from dataclasses import dataclass, asdict

# --- CONFIGURATION ---
# Define your backbone section here (or load from file)
BACKBONE_SECTION = "AAGCAAGTAAAACCTCTACAAATGTGGTATTGGCCCATCTCTATCGGTATCGTAGCATAACCCCTTGGGGCCTCTAAACGGGTCTTGAGGGGTTTTTTGTGCCCCTCGGGCCGGATTGCTATCTACCGGCATTGGCGCAGAAAAAAATGCCTGATGCGACGCTGCGCGTCTTATACTCCCACATATGCCAGATTCAGCAACGGATACGGCTTCCCCAACTTGCCCACTTCCATACGTGTCCTCCTTACCAGAAATTTATCCTTAAGGTCGTCAGCTATCCTGCAGGCGATCTCTCGATTTCGATCAAGACATTCCTTTAATGGTCTTTTCTGGACACCACTAGGGGTCAGAAGTAGTTCATCAAACTTTCTTCCCTCCCTAATCTCATTGGTTACCTTGGGCTATCGAAACTTAATTAACCAGTCAAGTCAGCTACTTGGCGAGATCGACTTGTCTGGGTTTCGACTACGCTCAGAATTGCGTCAGTCAAGTTCGATCTGGTCCTTGCTATTGCACCCGTTCTCCGATTACGAGTTTCATTTAAATCATGTGAGCAAAAGGCCAGCAAAAGGCCAGGAACCGTAAAAAGGCCGCGTTGCTGGCGTTTTTCCATAGGCTCCGCCCCCCTGACGAGCATCACAAAAATCGACGCTCAAGTCAGAGGTGGCGAAACCCGACAGGACTATAAAGATACCAGGCGTTTCCCCCTGGAAGCTCCCTCGTGCGCTCTCCTGTTCCGACCCTGCCGCTTACCGGATACCTGTCCGCCTTTCTCCCTTCGGGAAGCGTGGCGCTTTCTCATAGCTCACGCTGTAGGTATCTCAGTTCGGTGTAGGTCGTTCGCTCCAAGCTGGGCTGTGTGCACGAACCCCCCGTTCAGCCCGACCGCTGCGCCTTATCCGGTAACTATCGTCTTGAGTCCAACCCGGTAAGACACGACTTATCGCCACTGGCAGCAGCCACTGGTAACAGGATTAGCAGAGCGAGGTATGTAGGCGGTGCTACAGAGTTCTTGAAGTGGTGGCCTAACTACGGCTACACTAGAAGAACAGTATTTGGTATCTGCGCTCTGCTGAAGCCAGTTACCTTCGGAAAAAGAGTTGGTAGCTCTTGATCCGGCAAACAAACCACCGCTGGTAGCGGTGGTTTTTTTGTTTGCAAGCAGCAGATTACGCGCAGAAAAAAAGGATCTCAAGAAGATCCTTTGATCTTTTCTACGGGGTCTGACGCTCAGTGGAACGAAAACTCACGTTAAGGGATTTTGGTCATGAGATTATCAAAAAGGATCTTCACCTAGATCCTTTTAAATTAAAAATGAAGTTTTAAATCAATCTAAAGTATATATGAGTAAACTTGGTCTGACAGTTACCAATGCTTAATCAGTGAGGCACCTATCTCAGCGATCTGTCTATTTCGTTCATCCATAGTTGCATTTAAATTTCCGAACTCTCCAAGGCCCTCGTCGGAAAATCTTCAAACCTTTCGTCCGATCCATCTTGCAGGCTACCTCTCGAACGAACTATCGCAAGTCTCTTGGCCGGCCTTGCGCCTTGGCTATTGCTTGGCAGCGCCTATCGCCAGGTATTACTCCAATCCCGAATATCCGAGATCGGGATCACCCGAGAGAAGTTCAACCTACATCCTCAATCCCGATCTATCCGAGATCCGAGGAATATCGAAATCGGGGCGCGCCTGGTGTACCGAGAACGATCCTCTCAGTGCGAGTCTCGACGATCCATATCGTTGCTTGGCAGTCAGCCAGTCGGAATCCAGCTTGGGACCCAGGAAGTCCAATCGTCAGATATTGTACTCAAGCCTGGTCACGGCAGCGTACCGATCTGTTTAAACCTAGATATTGATAGTCTGATCGGTCAACGTATAATCGAGTCCTAGCTTTTGCAAACATCTATCAAGAGACAGGATCAGCAGGAGGCTTTCGCATGAGTATTCAACATTTCCGTGTCGCCCTTATTCCCTTTTTTGCGGCATTTTGCCTTCCTGTTTTTGCTCACCCAGAAACGCTGGTGAAAGTAAAAGATGCTGAAGATCAGTTGGGTGCGCGAGTGGGTTACATCGAACTGGATCTCAACAGCGGTAAGATCCTTGAGAGTTTTCGCCCCGAAGAACGCTTTCCAATGATGAGCACTTTTAAAGTTCTGCTATGTGGCGCGGTATTATCCCGTATTGACGCCGGGCAAGAGCAACTCGGTCGCCGCATACACTATTCTCAGAATGACTTGGTTGAGTATTCACCAGTCACAGAAAAGCATCTTACGGATGGCATGACAGTAAGAGAATTATGCAGTGCTGCCATAACCATGAGTGATAACACTGCGGCCAACTTACTTCTGACAACGATTGGAGGACCGAAGGAGCTAACCGCTTTTTTGCACAACATGGGGGATCATGTAACTCGCCTTGATCGTTGGGAACCGGAGCTGAATGAAGCCATACCAAACGACGAGCGTGACACCACGATGCCTGTAGCAATGGCAACAACCTTGCGTAAACTATTAACTGGCGAACTACTTACTCTAGCTTCCCGGCAACAGTTGATAGACTGGATGGAGGCGGATAAAGTTGCAGGACCACTTCTGCGCTCGGCCCTTCCGGCTGGCTGGTTTATTGCTGATAAATCTGGAGCCGGTGAGCGTGGGTCTCGCGGTATCATTGCAGCACTGGGGCCAGATGGTAAGCCCTCCCGTATCGTAGTTATCTACACGACGGGGAGTCAGGCAACTATGGATGAACGAAATAGACAGATCGCTGAGATAGGTGCCTCACTGATTAAGCATTGGTAACCGATTCTAGGTGCATTGGCGCAGAAAAAAATGCCTGATGCGACGCTGCGCGTCTTATACTCCCACATATGCCAGATTCAGCAACGGATACGGCTTCCCCAACTTGCCCACTTCCATACGTGTCCTCCTTACCAGAAATTTATCCTTAAGATCCCGAATCGTTTAAACTCGACTCTGGCTCTATCGAATCTCCGTCGTTTCGAGCTTACGCGAACAGCCGTGGCGCTCATTTGCTCGTCGGGCATCGAATCTCGTCAGCTATCGTCAGCTTACCTTTTTGGCAGCGATCGCGGCTCCCGACATCTTGGACCATTAGCTCCACAGGTATCTTCTTCCCTCTAGTGGTCATAACAGCAGCTTCAGCTACCTCTCAATTCAAAAAACCCCTCAAGACCCGTTTAGAGGCCCCAAGGGGTTATGCTATCAATCGTTGCGTTACACACACAAAAAACCAACACACATCCATCTTCGATGGATAGCGATTTTATTATCTAACTGCTGATCGAGTGTAGCCAGATCTAGTAATCAATTACGGGGTCATTAGTTCATAGC".upper()

@dataclass
class AlignmentStats:
    read_id: str
    read_len: int
    ref_start: int
    ref_end: int
    strand: int      # 1 for forward, -1 for reverse
    matches: int     # Number of matching bases
    block_len: int   # Length of the alignment block on the reference
    nm: int          # Edit distance (mismatches + gaps)
    error_rate: float
    is_valid: bool   # Flag if you want to filter later (e.g. alignment too short)

def estimate_error_rates(fastq_path, reference_seq, min_length=3000):
    """
    Aligns reads to a specific backbone sequence and calculates error rates.
    """
    print(f"Building index for reference ({len(reference_seq)} bp)...", file=sys.stderr)
    
    # 1. Create Aligner from the string directly
    # 'preset="map-ont"' is optimized for Nanopore reads
    aligner = mp.Aligner(seq=reference_seq, preset="map-ont")
    
    if not aligner:
        raise ValueError("Failed to build index. Is the sequence empty?")

    stats_storage = []
    
    print(f"Processing {fastq_path}...", file=sys.stderr)
    
    for name, seq, qual in mp.fastx_read(fastq_path):
        if len(seq) < min_length:
            continue
        
        # Align read to the backbone section
        # mappy.map() returns a generator of alignments
        hits = list(aligner.map(seq))
        
        if not hits:
            continue

        # 2. Find Best Single Alignment
        # Strategy: Sort by 'mlen' (matched length) descending. 
        # The hit with the most matching bases is almost always the "primary" alignment.
        best_hit = sorted(hits, key=lambda x: x.mlen, reverse=True)[0]
        
        # 3. Calculate Stats
        # nm = Edit Distance (mismatches + insertions + deletions)
        # blen = Alignment block length on the reference
        # Error Rate = Edit Distance / (Reference Span)
        # (You could also use seq length, but ref span is standard for 'divergence')
        
        if best_hit.blen == 0: continue

        err_rate = best_hit.NM / best_hit.blen
        
        # Store in RAM
        stat = AlignmentStats(
            read_id=name,
            read_len=len(seq),
            ref_start=best_hit.r_st,
            ref_end=best_hit.r_en,
            strand=best_hit.strand,
            matches=best_hit.mlen,
            block_len=best_hit.blen,
            nm=best_hit.NM,
            error_rate=err_rate,
            is_valid=True
        )
        
        stats_storage.append(stat)

    return stats_storage

# --- ANALYSIS HELPER ---
def analyze_results(stats_list):
    if not stats_list:
        print("No alignments found.")
        return

    # Filter for valid alignments only
    valid_stats = [s for s in stats_list if s.is_valid]
    
    if not valid_stats:
        print("No valid alignments passed the length filter.")
        return

    # Extract error rates
    errors = [s.error_rate for s in valid_stats]
    
    print("\n--- ERROR RATE ANALYSIS ---")
    print(f"Total Reads Aligned: {len(stats_list)}")
    print(f"Valid Alignments (>200bp): {len(valid_stats)}")
    print(f"Mean Error Rate:   {statistics.mean(errors):.2%}")
    print(f"Median Error Rate: {statistics.median(errors):.2%}")
    print(f"Min Error Rate:    {min(errors):.2%}")
    print(f"Max Error Rate:    {max(errors):.2%}")

# --- EXAMPLE USAGE ---
if __name__ == "__main__":
    # Example Inputs
    fastq_file = "/Users/ogw/Downloads/ris_plasmids/no_sample_id/20251208_1553_MN41644_AYO707_6ec906fc/fastq_pass/combined.fastq.gz"
    fastq_file = '/Users/ogw/Library/CloudStorage/GoogleDrive-oscargwilkins@gmail.com/My Drive/UCL PhD/2025/plasmid_sequencing_results/21641/2025-10-30_01-47-43/downstream_risdiplam_array_pool/downstream_risdiplam_array_pool_raw.fastq.gz'
    # Run
    stats = estimate_error_rates(fastq_file, BACKBONE_SECTION)
    analyze_results(stats)


Building index for reference (3353 bp)...
Processing /Users/ogw/Library/CloudStorage/GoogleDrive-oscargwilkins@gmail.com/My Drive/UCL PhD/2025/plasmid_sequencing_results/21641/2025-10-30_01-47-43/downstream_risdiplam_array_pool/downstream_risdiplam_array_pool_raw.fastq.gz...



--- ERROR RATE ANALYSIS ---
Total Reads Aligned: 1456
Valid Alignments (>200bp): 1456
Mean Error Rate:   1.47%
Median Error Rate: 0.95%
Min Error Rate:    0.00%
Max Error Rate:    15.03%


In [16]:
import edlib
import mappy as mp
import sys
from collections import namedtuple, Counter, defaultdict

# --- CONFIGURATION ---
try:
    sys.set_int_max_str_digits(0) # Allow massive integers for inserts
except AttributeError:
    pass

DNA_TO_DIGITS = str.maketrans("ACGTN", "01234")
DIGITS_TO_DNA = {0: 'A', 1: 'C', 2: 'G', 3: 'T', 4: 'N'}

# --- COMPRESSION UTILS ---
def compress_dna(seq_str):
    if not seq_str: return None
    return int(seq_str.translate(DNA_TO_DIGITS), 5)

def decompress_dna(val, length):
    if val is None: return "NA"
    chars = []
    for _ in range(length):
        remainder = val % 5
        chars.append(DIGITS_TO_DNA[remainder])
        val //= 5
    return "".join(reversed(chars))

# --- ALIGNMENT LOGIC ---
Hit = namedtuple('Hit', ['r_st', 'r_en', 'strand', 'score'])



def get_best_hit_edlib(target_seq, query_seq, query_rc, threshold_pct=0.20):
    best_hit = None
    best_ed = float('inf')
    candidates = [(query_seq, 1), (query_rc, -1)]
    max_dist = int(len(query_seq) * threshold_pct)

    for seq, strand_val in candidates:
        result = edlib.align(seq, target_seq, mode="HW", task="locations", k=max_dist)
        if result['editDistance'] == -1: continue 

        if result['editDistance'] < best_ed:
            best_ed = result['editDistance']
            loc = result['locations'][0]
            best_hit = Hit(loc[0], loc[1] + 1, strand_val, result['editDistance'])
            
    return best_hit

def extract_region(read_seq, f5, f5_rc, f3, f3_rc, max_len=10000):
    h5 = get_best_hit_edlib(read_seq, f5, f5_rc)
    h3 = get_best_hit_edlib(read_seq, f3, f3_rc)

    if not h5 or not h3:
        if not h5 and not h3: return None, "Both Flanks Missing"
        if not h5: return None, "5' Flank Missing"
        if not h3: return None, "3' Flank Missing"

    if h5.strand != h3.strand: return None, "Flank orientation mismatch"

    if h5.strand == 1:
        if h5.r_en >= h3.r_st: return None, "Overlap/Swap"
        dist = h3.r_st - h5.r_en
        if dist > max_len: return None, "Region too long"
        return read_seq[h5.r_en : h3.r_st], "PASS"
    else:
        if h3.r_en >= h5.r_st: return None, "RC Overlap/Swap"
        dist = h5.r_st - h3.r_en
        if dist > max_len: return None, "Region too long"
        return mp.revcomp(read_seq[h3.r_en : h5.r_st]), "PASS"

# --- MAIN PIPELINE ---
def process_fastq(fastq_path, bc_5p, bc_3p, ins_5p, ins_3p, min_length_fastq=1000):
    print(f"Processing: {fastq_path} (Inverted Index Mode)...", file=sys.stderr)
    
    bc_5p_rc = mp.revcomp(bc_5p)
    bc_3p_rc = mp.revcomp(bc_3p)
    ins_5p_rc = mp.revcomp(ins_5p)
    ins_3p_rc = mp.revcomp(ins_3p)

    bc_stats = Counter()
    ins_stats = Counter()
    total_processed = 0
    
    # --- NEW STORAGE STRUCTURE ---
    # 1. A simple list of compressed inserts. Access by index.
    inserts_list = [] 
    
    # 2. A dictionary mapping Barcode -> List of Indices in inserts_list
    # Example: { "ATGC": [0, 5, 12], "NA": [1, 3] }
    barcode_map = defaultdict(list)

    passed_hashes = set()
    
    for name, seq, qual in mp.fastx_read(fastq_path):
        if len(seq) < min_length_fastq: continue
        total_processed += 1
        
        bc_seq, bc_stat = extract_region(seq, bc_5p, bc_5p_rc, bc_3p, bc_3p_rc, max_len=100)
        ins_seq, ins_stat = extract_region(seq, ins_5p, ins_5p_rc, ins_3p, ins_3p_rc, max_len=10000)

        # Simplify Stats
        if bc_stat and "Region too long" in bc_stat: bc_stat = "Region too long"
        if ins_stat and "Region too long" in ins_stat: ins_stat = "Region too long"
        bc_stats[bc_stat] += 1
        ins_stats[ins_stat] += 1

        if total_processed % 5000 == 0:
            print(f"\rProcessed {total_processed} reads...", end="", file=sys.stderr)
        
        # --- STORAGE LOGIC ---
        if not bc_seq or not ins_seq:
            continue

        passed_hashes.add(hash(name))
        
        # 1. Compress Insert
        ins_comp = compress_dna(ins_seq)
        ins_len = len(ins_seq) if ins_seq else 0
        
        # 2. Append to Master List
        inserts_list.append((ins_comp, ins_len))
        
        # 3. Get the Index of the item we just added
        current_index = len(inserts_list) - 1
        
        # 4. Update Index (Dictionary)
        bc_key = bc_seq
        barcode_map[bc_key].append(current_index)
        


    print(f"\rDone. Processed {total_processed} reads.      ", file=sys.stderr)

    # Print Summary
    def print_table(title, stats):
        print(f"\n=== {title} SUMMARY ===")
        for k, v in stats.most_common():
            print(f"{k:<30} | {v:<8} | {(v/total_processed)*100:.1f}%")

    if total_processed > 0:
        print_table("BARCODE", bc_stats)
        print_table("INSERT", ins_stats)

    # ... existing code (after print_table calls) ...
    
    # NEW: Second pass to write failed reads
    fail_file = "/Users/ogw/Downloads/failed_reads.fastq"
    print(f"Writing failed reads to {fail_file}...", file=sys.stderr)
    
    with open(fail_file, "w") as out_f:
        for name, seq, qual in mp.fastx_read(fastq_path):
            if len(seq) < min_length_fastq:
                continue
            if hash(name) not in passed_hashes:
                out_f.write(f"@{name}\n{seq}\n+\n{qual}\n")

    return inserts_list, barcode_map
        
    return inserts_list, barcode_map

if __name__ == "__main__":
    barcode_5p = "AATAGGACGAgACGCGC".upper()
    barcode_3p = "cGTAAACTGGATCCGC".upper()
    insert_5p  = "CTTGGTGCCAGCTTATCA".upper()
    insert_3p  = "cctatgaagtgctctagtcaagtttaact".upper()

    fastq_file = "/Users/ogw/Downloads/ris_plasmids/no_sample_id/20251208_1553_MN41644_AYO707_6ec906fc/fastq_pass/combined.fastq.gz" 
    fastq_file = '/Users/ogw/Library/CloudStorage/GoogleDrive-oscargwilkins@gmail.com/My Drive/UCL PhD/2025/plasmid_sequencing_results/21641/2025-10-30_01-47-43/downstream_risdiplam_array_pool/downstream_risdiplam_array_pool_raw.fastq.gz'
    
    # Run
    inserts, bc_index = process_fastq(fastq_file, barcode_5p, barcode_3p, insert_5p, insert_3p)

    print(f"\rTotal number of good barcode/insert pairs: {len(inserts)}")
    
    # --- DEMO: HOW TO USE THE NEW STRUCTURE ---
    print("\n--- DATA ACCESS DEMO ---")
    
    # 1. Find the most common barcode
    # Sort barcodes by how many reads they have (length of the index list)
    sorted_bcs = sorted(bc_index.items(), key=lambda item: len(item[1]), reverse=True)
    
    if sorted_bcs:
        top_bc, indices = sorted_bcs[0]
        print(f"Top Barcode: {top_bc} (Count: {len(indices)})")
        
        # 2. Get the first 3 inserts associated with this top barcode
        print(f"First 3 inserts for {top_bc}:")
        for idx in indices[:3]:
            comp_ins, ins_len = inserts[idx]
            raw_ins = decompress_dna(comp_ins, ins_len)
            print(f"  - Index {idx}: {raw_ins[:40]}...")

# --- STATISTICS MODULE ---
    import statistics

    print("\n" + "="*30)
    print("   LENGTH STATISTICS (GOATED)   ")
    print("="*30)

    # 1. INSERT STATISTICS
    if inserts:
        ins_lengths = sorted([length for _, length in inserts]) # Sort once for percentiles
        n = len(ins_lengths)
        
        # Helper to get percentile (p is 0-100)
        def get_p(p): return ins_lengths[int((n - 1) * p / 100)]

        print(f"\n[INSERTS] (n={n})")
        print(f"  Mean: {statistics.mean(ins_lengths):.1f} bp")
        
        # requested quantiles
        print(f"  Min length: {min(ins_lengths)} bp")
        print(f"  Q1:   {get_p(1)} bp")
        print(f"  Q10:  {get_p(10)} bp")
        print(f"  Q25:  {get_p(25)} bp")
        print(f"  Q50:  {get_p(50)} bp (Median)")
        print(f"  Q75:  {get_p(75)} bp")
        print(f"  Q90:  {get_p(90)} bp")
        print(f"  Q99:  {get_p(99)} bp")
        print(f"  Max length: {max(ins_lengths)} bp")
        
    else:
        print("\n[INSERTS] No valid inserts found.")

    # 2. BARCODE STATISTICS (Weighted by Read Count)
    if bc_index:
        # Reconstruct list of lengths for every single read
        bc_lengths = []
        for bc_seq, index_list in bc_index.items():
            # Add the length of this barcode 'N' times, where N is how many reads had it
            bc_lengths.extend([len(bc_seq)] * len(index_list))
            
        print(f"\n[BARCODES] (n={len(bc_lengths)})")
        print(f"  Mean:   {statistics.mean(bc_lengths):.1f} bp")
        bc_med = statistics.median(bc_lengths)
        bc_med_pct = (bc_lengths.count(bc_med) / len(bc_lengths)) * 100
        print(f"  Median: {bc_med} bp ({bc_med_pct:.1f}% of reads match)")
        print(f"  Min:    {min(bc_lengths)} bp")
        print(f"  Max:    {max(bc_lengths)} bp")
    else:
        print("\n[BARCODES] No valid barcodes found.")
    
    print("\n" + "="*30)

Processing: /Users/ogw/Library/CloudStorage/GoogleDrive-oscargwilkins@gmail.com/My Drive/UCL PhD/2025/plasmid_sequencing_results/21641/2025-10-30_01-47-43/downstream_risdiplam_array_pool/downstream_risdiplam_array_pool_raw.fastq.gz (Inverted Index Mode)...



=== BARCODE SUMMARY ===
PASS                           | 1394     | 82.8%
Both Flanks Missing            | 172      | 10.2%
5' Flank Missing               | 74       | 4.4%
3' Flank Missing               | 40       | 2.4%
Flank orientation mismatch     | 1        | 0.1%
RC Overlap/Swap                | 1        | 0.1%
Region too long                | 1        | 0.1%

=== INSERT SUMMARY ===
PASS                           | 1296     | 77.0%
3' Flank Missing               | 173      | 10.3%
Both Flanks Missing            | 165      | 9.8%
5' Flank Missing               | 30       | 1.8%
RC Overlap/Swap                | 10       | 0.6%
Overlap/Swap                   | 9        | 0.5%
Total number of good barcode/insert pairs: 1219

--- DATA ACCESS DEMO ---
Top Barcode: CCACTTTACTTCCA (Count: 4)
First 3 inserts for CCACTTTACTTCCA:
  - Index 80: GCGCGTCCTGGTAAGTCTCATGAGTAAGAAAGAGTAAGAG...
  - Index 341: GCGCGTCCTGGTAAGTCTCATGAGTAAGAAAGAGTAAGAG...
  - Index 801: GCGCGTCCTGGTAAGTCTCATGAGTAAGA

Done. Processed 1683 reads.      
Writing failed reads to /Users/ogw/Downloads/failed_reads.fastq...


In [17]:
print(bc_index)

defaultdict(<class 'list'>, {'ACAAACCAACCCCT': [0], 'AACCAAAAATTTCC': [1, 908], 'CCGACTTTCTCCCCC': [2], 'CCATCACAATTATA': [3], 'TCTCATATCCACTT': [4, 345], 'AATATATATATCAA': [5], 'TCTTACTTATCCCT': [6], 'CCCACTCTATTCCC': [7], 'CATCCTCTCCATAT': [8, 1091], 'TATCACCTCACTCA': [9], 'AACCACATCTTCTA': [10], 'ATTTTCCTCCATCC': [11], 'AACCTCCAACACAT': [12], 'CCCCCTAAAACTCA': [13], 'ATTCCACACACAAC': [14], 'TCTCCCTTAAATTA': [15], 'TTACCAACACTCCA': [16], 'TATCCCATCTCCTA': [17], 'CCACAAATCCTG': [18], 'TATTTTACCACCTC': [19], 'ATAAATCTATTCAA': [20], 'AATTATTTTCCCA': [21], 'CTATTACCCCTCAT': [22, 24, 523], 'TTACACCCCCTCTC': [23], 'CCCCCAACCCCCCC': [25, 298], 'TACCCAACATAAAT': [26], 'CCCTACTCCTAACA': [27], 'CCTACCCCTCACAT': [28], 'TATTTTACTTAATA': [29], 'ATTCTCATCCCCCC': [30], 'TACTAAATAAACAA': [31], 'TAAACTTTATCATC': [32], 'ATAACCCCAAACAA': [33, 1081], 'ACTTTATCATTACA': [34], 'CAAATAATTTCTAC': [35], 'TCCCTTTCCCCTTA': [36], 'CCACTCAACCTTAA': [37], 'TATATCAACCATCC': [38], 'CTTCCACACTTATA': [39], 'TACTTCAATC